# 02 – Huấn luyện PhoBERT & so sánh với baseline

**Quy tắc chống rò rỉ dữ liệu**

| Tập | Dùng để |
|---|---|
| train | cập nhật trọng số, tính class weights |
| validation | early stopping (chọn epoch theo Macro-F1) + tìm ngưỡng lớp Trung tính |
| test | **chỉ đánh giá 1 lần ở cuối** |

Khuyến nghị chạy trên GPU (Colab/Kaggle T4: ~3–4 phút/epoch). Có thể chạy tương đương bằng lệnh `python -m src.train`.

In [ ]:
# Cho phép import src/ dù mở notebook từ thư mục notebooks/
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, resolve
cfg = load_config()
print("Project root:", ROOT)

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
cfg["training"]

## 1. Baseline TF-IDF (CPU, vài giây)

In [ ]:
from src.baseline import run as run_baseline
baseline = run_baseline(cfg)
baseline.round(4)

## 2. Fine-tune PhoBERT

In [ ]:
from src.train import train
summary = train(cfg)   # lưu checkpoint tốt nhất vào models/phobert-sentiment/
summary

In [ ]:
import pandas as pd
from IPython.display import Image, display
hist = pd.read_csv(resolve(cfg["results"]["metrics_dir"]) / "training_history.csv")
display(hist.round(4))
display(Image(resolve(cfg["results"]["figures_dir"]) / "loss_curves.png"))

In [ ]:
fig_dir = resolve(cfg["results"]["figures_dir"])
for tag in ("argmax", "threshold"):
    p = fig_dir / f"phobert_cm_test_{tag}.png"
    if p.exists():
        display(Image(p))

## 3. Bảng so sánh tổng hợp (trên TEST)

In [ ]:
import json
import matplotlib.pyplot as plt

metrics_dir = resolve(cfg["results"]["metrics_dir"])
rows = [{"model": r["model"], "accuracy": r["test_accuracy"], "macro_f1": r["test_macro_f1"]}
        for _, r in pd.read_csv(metrics_dir / "baseline_summary.csv").iterrows()]
ph = json.load(open(metrics_dir / "phobert_test_summary.json", encoding="utf-8"))
for tag, m in ph.items():
    rows.append({"model": f"phobert_{tag}", "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]})
comp = pd.DataFrame(rows).set_index("model")
comp.to_csv(metrics_dir / "final_comparison.csv")
display(comp.round(4))

ax = comp.plot.barh(figsize=(8, 4), xlim=(0.5, 1.0))
ax.set_title("So sánh mô hình trên tập test"); plt.tight_layout()
plt.savefig(fig_dir / "overall_comparison_chart.png", dpi=150); plt.show()